In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("UrgenceNet - Partie 3 - DataFrames et SQL")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.memory", "1g")
    .config("spark.executor.cores", "1")
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4")
    .config("spark.jars.ivy", "/home/spark/.ivy2")
    .config("spark.hadoop.fs.s3a.endpoint", "minio:9000")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.access.key", "urgenceadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "urgence-local-2026")
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
    )
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

sc = spark.sparkContext
print("Spark :", spark.version)
print("Master :", sc.master)

Spark : 3.5.7
Master : spark://spark-master:7077


In [3]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

event_schema = T.StructType([
    T.StructField("visit_id", T.StringType()),
    T.StructField("patient_id", T.StringType()),
    T.StructField("facility_id", T.StringType()),
    T.StructField("city", T.StringType()),
    T.StructField("status", T.StringType()),
    T.StructField("priority", T.IntegerType()),
    T.StructField("wait_minutes", T.IntegerType()),
    T.StructField("department", T.StringType()),
    T.StructField("desk_id", T.StringType()),
    T.StructField("transfer_to", T.StringType()),
    T.StructField("event_time", T.TimestampType()),
])

facility_schema = T.StructType([
    T.StructField("facility_id", T.StringType()),
    T.StructField("name", T.StringType()),
    T.StructField("city", T.StringType()),
    T.StructField("hospital_group", T.StringType()),
    T.StructField("type", T.StringType()),
    T.StructField("beds", T.IntegerType()),
    T.StructField("latitude", T.DoubleType()),
    T.StructField("longitude", T.DoubleType()),
])

history_path = "s3a://lake/bronze/history/urgences_history.jsonl"
facilities_path = "s3a://lake/bronze/reference/facilities.json"

events_df = spark.read.schema(event_schema).json(history_path)
facilities_df = spark.read.schema(facility_schema).json(facilities_path)

events_df.printSchema()
print("Événements :", events_df.count())
print("Établissements :", facilities_df.count())
facilities_df.select("facility_id", "name", "type").show(5, truncate=False)

26/11/19 15:52:06 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


root
 |-- visit_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- facility_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- status: string (nullable = true)
 |-- priority: integer (nullable = true)
 |-- wait_minutes: integer (nullable = true)
 |-- department: string (nullable = true)
 |-- desk_id: string (nullable = true)
 |-- transfer_to: string (nullable = true)
 |-- event_time: timestamp (nullable = true)



Événements : 50000
Établissements : 40
+-----------+--------------------------------+----------------+
|facility_id|name                            |type            |
+-----------+--------------------------------+----------------+
|F-001      |Casablanca - CHU 01             |CHU             |
|F-002      |Casablanca - HOPITAL_REGIONAL 02|HOPITAL_REGIONAL|
|F-003      |Casablanca - HOPITAL_REGIONAL 03|HOPITAL_REGIONAL|
|F-004      |Casablanca - HOPITAL_REGIONAL 04|HOPITAL_REGIONAL|
|F-005      |Casablanca - CLINIQUE 05        |CLINIQUE        |
+-----------+--------------------------------+----------------+
only showing top 5 rows



In [4]:
care_df = events_df.filter(F.col("status") == "CARE_STARTED")

city_df = (
    care_df.groupBy("city")
    .agg(
        F.count("*").alias("prises_en_charge"),
        F.avg("wait_minutes").alias("attente_moyenne_min"),
    )
    .orderBy("city")
)

city_df.show()

[Stage 7:=============================>                             (1 + 1) / 2]

+----------+----------------+-------------------+
|      city|prises_en_charge|attente_moyenne_min|
+----------+----------------+-------------------+
|Casablanca|            3079| 47.609288730107174|
| Marrakech|            3194| 47.246399499060736|
|     Rabat|            3144|  47.30089058524173|
|    Tanger|            3122|  47.93818065342729|
+----------+----------------+-------------------+



In [5]:
events_df.createOrReplaceTempView("urgences")

city_sql = spark.sql("""
    SELECT
        city,
        COUNT(*) AS prises_en_charge,
        AVG(wait_minutes) AS attente_moyenne_min
    FROM urgences
    WHERE status = 'CARE_STARTED'
    GROUP BY city
    ORDER BY city
""")

city_sql.show()

assert city_df.collect() == city_sql.collect(), "Les deux résultats diffèrent"

+----------+----------------+-------------------+
|      city|prises_en_charge|attente_moyenne_min|
+----------+----------------+-------------------+
|Casablanca|            3079| 47.609288730107174|
| Marrakech|            3194| 47.246399499060736|
|     Rabat|            3144|  47.30089058524173|
|    Tanger|            3122|  47.93818065342729|
+----------+----------------+-------------------+



In [6]:
facility_lookup = facilities_df.select(
    "facility_id",
    F.col("type").alias("facility_type"),
    "hospital_group",
)

care_with_facility = care_df.join(
    F.broadcast(facility_lookup),
    on="facility_id",
    how="left",
)

sc.setLocalProperty("spark.job.description", "DataFrame - jointure établissements")
try:
    (
        care_with_facility.groupBy("city", "facility_type")
        .agg(
            F.count("*").alias("prises_en_charge"),
            F.avg("wait_minutes").alias("attente_moyenne_min"),
        )
        .orderBy("city", "facility_type")
        .show(30, truncate=False)
    )
finally:
    sc.setLocalProperty("spark.job.description", None)

care_with_facility.explain(True)

[Stage 30:>                                                         (0 + 2) / 2]

+----------+----------------+----------------+-------------------+
|city      |facility_type   |prises_en_charge|attente_moyenne_min|
+----------+----------------+----------------+-------------------+
|Casablanca|CENTRE_SANTE    |908             |48.284140969163    |
|Casablanca|CHU             |299             |47.98996655518395  |
|Casablanca|CLINIQUE        |935             |46.959358288770055 |
|Casablanca|HOPITAL_REGIONAL|937             |47.48239060832444  |
|Marrakech |CENTRE_SANTE    |999             |47.46446446446446  |
|Marrakech |CHU             |291             |48.38831615120275  |
|Marrakech |CLINIQUE        |952             |47.21008403361345  |
|Marrakech |HOPITAL_REGIONAL|952             |46.70483193277311  |
|Rabat     |CENTRE_SANTE    |929             |48.01722282023681  |
|Rabat     |CHU             |329             |47.58358662613982  |
|Rabat     |CLINIQUE        |953             |47.335781741867784 |
|Rabat     |HOPITAL_REGIONAL|933             |46.4523043944265

In [7]:
@F.udf(T.StringType())
def wait_band(wait_minutes):
    if wait_minutes is None:
        return None
    if wait_minutes < 15:
        return "court"
    if wait_minutes <= 45:
        return "moyen"
    return "long"


minutes = F.col("wait_minutes")

wait_band_native = (
    F.when(minutes.isNull(), F.lit(None).cast("string"))
    .when(minutes < 15, "court")
    .when(minutes <= 45, "moyen")
    .otherwise("long")
)

bands = care_df.select(
    "wait_minutes",
    wait_band(minutes).alias("bande_udf"),
    wait_band_native.alias("bande_native"),
)

bands.show(10)

differences = bands.filter(
    ~F.col("bande_udf").eqNullSafe(F.col("bande_native"))
)

print("Différences entre les deux méthodes :", differences.count())
bands.explain()

+------------+---------+------------+
|wait_minutes|bande_udf|bande_native|
+------------+---------+------------+
|           9|    court|       court|
|          11|    court|       court|
|          17|    moyen|       moyen|
|          37|    moyen|       moyen|
|          28|    moyen|       moyen|
|          17|    moyen|       moyen|
|          44|    moyen|       moyen|
|          12|    court|       court|
|          14|    court|       court|
|           7|    court|       court|
+------------+---------+------------+
only showing top 10 rows



[Stage 34:=============================>                            (1 + 1) / 2]

Différences entre les deux méthodes : 0
== Physical Plan ==
*(2) Project [wait_minutes#6, pythonUDF0#293 AS bande_udf#261, CASE WHEN isnull(wait_minutes#6) THEN null WHEN (wait_minutes#6 < 15) THEN court WHEN (wait_minutes#6 <= 45) THEN moyen ELSE long END AS bande_native#262]
+- BatchEvalPython [wait_band(wait_minutes#6)#260], [pythonUDF0#293]
   +- *(1) Project [wait_minutes#6]
      +- *(1) Filter (isnotnull(status#4) AND (status#4 = CARE_STARTED))
         +- FileScan json [status#4,wait_minutes#6] Batched: false, DataFilters: [isnotnull(status#4), (status#4 = CARE_STARTED)], Format: JSON, Location: InMemoryFileIndex(1 paths)[s3a://lake/bronze/history/urgences_history.jsonl], PartitionFilters: [], PushedFilters: [IsNotNull(status), EqualTo(status,CARE_STARTED)], ReadSchema: struct<status:string,wait_minutes:int>




In [8]:
clean_df = events_df.dropDuplicates(["visit_id", "status"])

before = events_df.count()
after = clean_df.count()

print("Avant :", before)
print("Après :", after)
print("Doublons supprimés :", before - after)

[Stage 40:>                                                         (0 + 2) / 2]

Avant : 50000
Après : 49000
Doublons supprimés : 1000


In [9]:
arrivals = clean_df.filter(F.col("status") == "ARRIVED").select(
    "visit_id",
    F.col("event_time").alias("arrival_time"),
)

care_times = clean_df.filter(F.col("status") == "CARE_STARTED").select(
    "visit_id",
    "wait_minutes",
    F.col("event_time").alias("care_time"),
)

wait_check = care_times.join(arrivals, on="visit_id", how="left").withColumn(
    "wait_recalculated",
    (F.col("care_time").cast("long") - F.col("arrival_time").cast("long")) / 60,
)

wait_check.show(10, truncate=False)

invalid_waits = wait_check.filter(
    F.col("arrival_time").isNull()
    | F.col("care_time").isNull()
    | F.col("wait_minutes").isNull()
    | (F.col("wait_recalculated") != F.col("wait_minutes"))
)

print("Prises en charge contrôlées :", wait_check.count())
print("Délais incohérents ou incomplets :", invalid_waits.count())

+---------+------------+-------------------+-------------------+-----------------+
|visit_id |wait_minutes|care_time          |arrival_time       |wait_recalculated|
+---------+------------+-------------------+-------------------+-----------------+
|V-0000412|75          |2026-11-19 15:32:10|2026-11-19 14:17:10|75.0             |
|V-0006096|59          |2026-11-19 16:19:03|2026-11-19 15:20:03|59.0             |
|V-0012059|12          |2026-11-19 18:47:35|2026-11-19 18:35:35|12.0             |
|V-0002697|73          |2026-11-19 23:12:04|2026-11-19 21:59:04|73.0             |
|V-0002757|58          |2026-11-20 02:25:29|2026-11-20 01:27:29|58.0             |
|V-0007902|61          |2026-11-20 03:56:00|2026-11-20 02:55:00|61.0             |
|V-0010721|85          |2026-11-20 05:24:19|2026-11-20 03:59:19|85.0             |
|V-0000887|83          |2026-11-20 07:09:56|2026-11-20 05:46:56|83.0             |
|V-0009370|10          |2026-11-20 08:43:06|2026-11-20 08:33:06|10.0             |
|V-0

Prises en charge contrôlées : 12250


[Stage 66:>                                                         (0 + 1) / 1]

Délais incohérents ou incomplets : 0


In [10]:
silver_path = "s3a://lake/silver/urgences_history/"

(
    clean_df.write
    .mode("overwrite")
    .partitionBy("city")
    .parquet(silver_path)
)

silver_df = spark.read.parquet(silver_path)

print("Événements relus dans Silver :", silver_df.count())
silver_df.groupBy("city").count().orderBy("city").show()

Événements relus dans Silver : 49000
+----------+-----+
|      city|count|
+----------+-----+
|Casablanca|12004|
| Marrakech|12436|
|     Rabat|12276|
|    Tanger|12284|
+----------+-----+



In [11]:
import json
from statistics import median
from time import perf_counter


def run_rdd():
    totals = (
        sc.textFile(history_path)
        .map(json.loads)
        .filter(lambda event: event["status"] == "CARE_STARTED")
        .map(lambda event: (event["city"], (event["wait_minutes"], 1)))
        .reduceByKey(
            lambda left, right: (left[0] + right[0], left[1] + right[1])
        )
        .collect()
    )
    return sorted(
        (city, count, total_wait / count)
        for city, (total_wait, count) in totals
    )


def run_dataframe():
    rows = (
        spark.read.schema(event_schema).json(history_path)
        .filter(F.col("status") == "CARE_STARTED")
        .groupBy("city")
        .agg(F.count("*").alias("count"), F.avg("wait_minutes").alias("average"))
        .collect()
    )
    return sorted((row["city"], row["count"], row["average"]) for row in rows)


assert run_rdd() == run_dataframe(), "Les résultats du benchmark diffèrent"

timings = {"RDD": [], "DataFrame": []}

for trial in range(3):
    runs = [("RDD", run_rdd), ("DataFrame", run_dataframe)]
    if trial % 2 == 1:
        runs.reverse()

    for label, operation in runs:
        start = perf_counter()
        operation()
        timings[label].append(perf_counter() - start)

for label, durations in timings.items():
    print(f"{label} | essais : {[round(t, 3) for t in durations]}")
    print(f"{label} | médiane : {median(durations):.3f} s")

RDD | essais : [0.771, 0.673, 0.857]
RDD | médiane : 0.771 s
DataFrame | essais : [0.669, 0.435, 0.531]
DataFrame | médiane : 0.531 s
